# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library.

### Dataset Source
The dataset schema is described using the [Croissant](https://mlcommons.org/croissant/) format and is accessible via the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading

Load the metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL for the dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset Name:", metadata.name)
print("Description:", metadata.description)

## 2. Data Overview

Let's review the available record sets and their fields using their `@id` values. This overview helps in identifying which data tables and columns can be accessed for analysis.

In [ ]:
# List all record sets available in the dataset, referenced by their @id
record_sets = list(dataset.record_sets)
print(f"Total record sets: {len(record_sets)}")

for rs in record_sets:
    print(f"Record Set @id: {rs['@id']}")
    # List fields/columns for this record set
    if 'field' in rs:
        print("  Fields:")
        for field in rs['field']:
            if isinstance(field, dict):
                field_id = field.get('@id', str(field))
            else:
                field_id = str(field)
            print(f"    - {field_id}")
    else:
        print("  No fields listed.")
    print()

## 3. Data Extraction

Extract data from one or more record sets using their `@id` fields. This section loads each record set into a Pandas DataFrame for further analysis.

Use the record set and field `@id` values obtained from the previous overview.

In [ ]:
# For illustration, extract all record sets. If none available, display a message.
all_dataframes = {}
record_set_ids = [rs['@id'] for rs in record_sets]

for rs_id in record_set_ids:
    print(f"Loading records from record set: {rs_id}")
    try:
        records = list(dataset.records(record_set=rs_id))
        if len(records) > 0:
            df = pd.DataFrame(records)
            all_dataframes[rs_id] = df
            print(f"  Loaded {len(df)} records. Columns: {df.columns.tolist()}")
        else:
            print("  No records found in this record set.")
    except Exception as e:
        print(f"  Error loading record set {rs_id}: {e}")

if len(all_dataframes) > 0:
    first_record_set = list(all_dataframes.keys())[0]
    print(f"\nFirst available record set: {first_record_set}")
    print("Columns:")
    print(all_dataframes[first_record_set].columns.tolist())
    display(all_dataframes[first_record_set].head())
else:
    print("No record sets with records were found in this dataset.")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records, normalizing numeric fields, or grouping data. For the example, we select the first available record set and apply sample EDA steps.

In [ ]:
# Demonstrate EDA if a dataframe is available
import numpy as np

if len(all_dataframes) > 0:
    df = all_dataframes[first_record_set]
    # Find numeric columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if len(numeric_cols) == 0:
        print("No numeric columns available for EDA.")
    else:
        numeric_field = numeric_cols[0]  # Use the first numeric field for demonstration
        print(f"Using numeric field '{numeric_field}' for EDA.")
        # Filter: values > threshold (using mean as threshold)
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records where {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized '{numeric_field}' for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Grouping: Try grouping by the first non-numeric column (if any)
        non_numeric_cols = [col for col in df.columns if col not in numeric_cols]
        if non_numeric_cols:
            group_field = non_numeric_cols[0]
            print(f"\nGrouping by field '{group_field}':")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            display(grouped_df.head())
        else:
            print("No non-numeric columns available for grouping.")
else:
    print("No record sets with data available for EDA.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset. Example plots demonstrate numeric field distributions or group-wise comparisons.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(all_dataframes) > 0 and len(df) > 0 and len(numeric_cols) > 0:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of '{numeric_field}'")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if non_numeric_cols:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=df[non_numeric_cols[0]], y=df[numeric_field])
        plt.xticks(rotation=45)
        plt.title(f"{numeric_field} by {non_numeric_cols[0]}")
        plt.xlabel(non_numeric_cols[0])
        plt.ylabel(numeric_field)
        plt.tight_layout()
        plt.show()
else:
    print("Insufficient data for visualization.")

## 6. Conclusion

In this notebook, we loaded and explored a FAIR² dataset package using `mlcroissant`. We examined the dataset's metadata, inspected available record sets, extracted data into DataFrames, performed essential exploratory data analysis, and visualized field distributions. 

For further analysis, refer to the variable and field `@id`s provided in the Croissant schema and apply domain-specific transformations or statistical methods as needed.